# Hey Pihu - OpenWakeWord Training Notebook
This notebook sets up a complete custom wake-word training pipeline for the wake phrase **'Hey Pihu'** using OpenWakeWord.

## 1. Install Dependencies

In [ ]:

!apt-get update -qq
!apt-get install -y espeak-ng ffmpeg

!pip install -U pip setuptools wheel

# Clone repositories
!git clone --branch v0.5.1 https://github.com/dscripka/openWakeWord.git
!git clone https://github.com/rhasspy/piper-sample-generator

# Install OpenWakeWord
!pip install -e ./openWakeWord

# Install dependencies
!pip install \
webrtcvad \
mutagen \
torchinfo \
torchmetrics \
speechbrain \
audiomentations \
torch-audiomentations \
acoustics \
pronouncing \
datasets \
deep-phonemizer \
pyyaml


## 2. Download Required Models

In [ ]:

import os

base = "./openWakeWord/openwakeword/resources/models"
os.makedirs(base, exist_ok=True)

files = {
    "embedding_model.onnx":
    "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx",

    "embedding_model.tflite":
    "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite",

    "melspectrogram.onnx":
    "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx",

    "melspectrogram.tflite":
    "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite",
}

for name, url in files.items():
    !wget -q $url -O {base}/{name}

print("Models downloaded.")


## 3. Create Training Config

In [ ]:

import yaml

config = {
    "model_name": "hey_pihu",

    "target_phrase": [
        "hey pihu",
        "okay pihu",
        "hello pihu",
        "hey peehoo",
        "hey pee who",
        "hey peewhoo"
    ],

    "custom_negative_phrases": [
        "hey people",
        "people",
        "peek who",
        "pee who",
        "hey siri",
        "okay google",
        "alexa",
        "computer"
    ],

    "n_samples": 5000,
    "n_samples_val": 1000,

    "tts_batch_size": 50,
    "augmentation_batch_size": 16,

    "piper_sample_generator_path":
        "./piper-sample-generator",

    "output_dir":
        "./hey_pihu_model",

    "rir_paths":
        ["./mit_rirs"],

    "background_paths":
        ["./background_clips"],

    "background_paths_duplication_rate":
        [1],

    "false_positive_validation_data_path":
        "./validation_set_features.npy",

    "augmentation_rounds": 1,

    "feature_data_files": {
        "ACAV100M_sample":
        "./openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
    },

    "batch_n_per_class": {
        "ACAV100M_sample": 1024,
        "adversarial_negative": 50,
        "positive": 50
    },

    "model_type": "dnn",

    "layer_size": 32,

    "steps": 30000,

    "max_negative_weight": 1500,

    "target_false_positives_per_hour": 0.2
}

with open("my_model.yaml", "w") as f:
    yaml.dump(config, f)

print("my_model.yaml created")


## 4. Generate Synthetic Clips

In [ ]:

import sys
import os

sys.path.append("/content/openWakeWord")
sys.path.append("/content/piper-sample-generator")

%cd /content/openWakeWord

!python openwakeword/train.py \
  --training_config ../my_model.yaml \
  --generate_clips


## 5. Train the Wake Word Model

In [ ]:

%cd /content/openWakeWord

!python openwakeword/train.py \
  --training_config ../my_model.yaml \
  --train_model


## 6. Add Your Own Audio
Upload your own WAV files (16kHz mono recommended) to improve accuracy.

Suggested structure:
```
custom_data/
├── positive/
├── negative/
```